# Filtered task-session table

Load the locally downloaded, pinned task and session tables. Keep tasks with nonempty session lists and states `cancelled`, `completed`, or `failed`. Keep sessions with `log_found = True` and the same allowed states, then inner-join on task ID.

The dataset spells canceled as `cancelled` and uses `log_found`, not `log-found`. Tasks without any qualifying sessions disappear from the join; individual excluded sessions do not disqualify their other sessions. Log availability is taken from the metadata flag; log contents are not downloaded or loaded. The result, `task_sessions`, contains only the requested fields, with task/session prefixes where names overlap.

In [14]:
from pathlib import Path

import duckdb
from agentlogs.schema import assert_dataset_version
from IPython.display import display

In [15]:
roots = [
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'packages' / 'risenlab-agentlogs').is_dir()
    and (path / 'data' / 'dataset-sample').is_dir()
]
if not roots:
    raise FileNotFoundError('Open this notebook from within the agentlogs repository.')

REPO_ROOT = roots[0]
RELEASE = '1.0-20261008'
REVISION = '25549ec57745fdf174620c1e75d7fd02d9c4dc22'
dataset_path = REPO_ROOT / 'data' / 'eda' / RELEASE / REVISION
assert_dataset_version(dataset_path)

con = duckdb.connect()
for table in ('agent_tasks', 'agent_sessions'):
    files = sorted((dataset_path / table).glob('*.parquet'))
    if not files:
        raise FileNotFoundError(
            f'Missing {table} Parquet files. Run eda/download_datasets.py first.'
        )
    con.read_parquet([str(path) for path in files]).create_view(table, replace=True)

In [16]:
con.execute('''
    CREATE OR REPLACE TEMP VIEW filtered_tasks AS
    SELECT *
    FROM agent_tasks
    WHERE array_length(sessions) > 0
      AND state IN ('cancelled', 'completed', 'failed')
''')

con.execute('''
    CREATE OR REPLACE TEMP VIEW filtered_sessions AS
    SELECT *
    FROM agent_sessions
    WHERE log_found IS TRUE
      AND state IN ('cancelled', 'completed', 'failed')
''');

In [17]:
task_sessions = con.sql('''
    SELECT
        t.id AS task_id,
        t.repository.full_name AS repository_full_name,
        t.creator.id AS creator_id,
        t.creator.login AS creator_login,
        t.creator.graphql_id AS creator_graphql_id,
        t.creator.type AS creator_type,
        t.name AS task_name,
        t.state AS task_state,
        t.created_at AS task_created_at,
        s.id AS session_id,
        s.session_index,
        s.name AS session_name,
        s.model,
        s.prompt,
        s.state AS session_state,
        s.usage.type AS usage_type,
        s.usage.amount AS usage_amount,
        s.usage.credits AS usage_credits,
        s.premium_requests,
        s.error,
        s.created_at AS session_created_at
    FROM filtered_tasks AS t
    INNER JOIN filtered_sessions AS s ON t.id = s.task.id
    ORDER BY t.id, s.session_index, s.id
''').fetchdf()

assert not task_sessions[['task_id', 'session_id']].isna().any().any(), 'Null join keys.'
assert not task_sessions.duplicated(['task_id', 'session_id']).any(), 'Duplicate task-session pairs.'
print(f'{len(task_sessions):,} task-session rows; {task_sessions.task_id.nunique():,} unique tasks.')
display(task_sessions.head())

631,141 task-session rows; 351,522 unique tasks.


,task_id,repository_full_name,creator_id,creator_login,creator_graphql_id,creator_type,task_name,task_state,task_created_at,session_id,...,session_name,model,prompt,session_state,usage_type,usage_amount,usage_credits,premium_requests,error,session_created_at
0,0000004c-372c-42a1-ac1c-9445fa268aa3,microsoft/testfx,5735905,nohwnd,MDQ6VXNlcjU3MzU5MDU=,user,Enhance Assert.AreEqual string comparison with...,completed,2025-07-31 15:29:13.263,1014564a-ea77-4371-8074-2ff6391d7242,...,Initial implementation,NaN,NaN,completed,premium_requests,1.0,1.0,1.0,NaN,2025-07-25 08:21:48.344
1,0000004c-372c-42a1-ac1c-9445fa268aa3,microsoft/testfx,5735905,nohwnd,MDQ6VXNlcjU3MzU5MDU=,user,Enhance Assert.AreEqual string comparison with...,completed,2025-07-31 15:29:13.263,add85a91-ec46-4147-a37c-6a2a03c3a5ba,...,Review from @Evangelink,NaN,NaN,completed,premium_requests,1.0,1.0,1.0,NaN,2025-07-25 09:15:22.748
2,0000004c-372c-42a1-ac1c-9445fa268aa3,microsoft/testfx,5735905,nohwnd,MDQ6VXNlcjU3MzU5MDU=,user,Enhance Assert.AreEqual string comparison with...,completed,2025-07-31 15:29:13.263,48dc4253-5818-4ec2-8e87-35ec3b41a222,...,Review from @Evangelink,NaN,NaN,completed,premium_requests,1.0,1.0,1.0,NaN,2025-07-25 10:51:24.037
3,0000004c-372c-42a1-ac1c-9445fa268aa3,microsoft/testfx,5735905,nohwnd,MDQ6VXNlcjU3MzU5MDU=,user,Enhance Assert.AreEqual string comparison with...,completed,2025-07-31 15:29:13.263,03aaf257-c3b3-434d-896e-d3458ba079c3,...,Review from @Evangelink,NaN,NaN,completed,premium_requests,1.0,1.0,1.0,NaN,2025-07-25 11:16:58.542
4,0000004c-372c-42a1-ac1c-9445fa268aa3,microsoft/testfx,5735905,nohwnd,MDQ6VXNlcjU3MzU5MDU=,user,Enhance Assert.AreEqual string comparison with...,completed,2025-07-31 15:29:13.263,4653054f-04d4-43c9-8bf9-5b809e20919f,...,Review from @Evangelink,NaN,NaN,completed,premium_requests,1.0,1.0,1.0,NaN,2025-07-25 14:41:09.729


In [28]:
import pandas as pd

# show me 10 rows where prompt is not null in task sessions
# order it by task_id and session_index both in ascending order
# do not truncate the prompt column
# repository_full_name should start with 'microsoft/'
# display the df and also write to csv to eda\sample_output
pd.set_option('display.max_colwidth', None)
df = con.sql('''
    SELECT *
    FROM task_sessions
    WHERE prompt IS NOT NULL
      AND repository_full_name LIKE 'microsoft/%'
    ORDER BY task_id ASC, session_index ASC
''').df()
display(df.head(10))
output_dir = REPO_ROOT / 'eda' / 'sample_output'
output_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(output_dir / 'task_sessions.csv', index=False)

,task_id,repository_full_name,creator_id,creator_login,creator_graphql_id,creator_type,task_name,task_state,task_created_at,session_id,...,session_name,model,prompt,session_state,usage_type,usage_amount,usage_credits,premium_requests,error,session_created_at
0,00155e12-992d-4f3a-a84e-072cd9263364,microsoft/bc-bench,27280733,haoranpb,MDQ6VXNlcjI3MjgwNzMz,user,Updating workflows to use latest setup-uv action,completed,2026-04-12 07:03:59.220,e27694f5-4061-42a2-ad3c-28095d457355,...,Updating workflows to use latest setup-uv action,sweagent-capi:claude-opus-4.6,Replace the setup-uv action in all workflows with the latest version `astral-sh/setup-uv@cec208311dfd045dd5311c1add060b2062131d57`,completed,premium_requests,3.000000e+00,3.000000e+00,3.0,NaN,2026-04-12 07:04:01.625
1,0031ce4b-a8f0-45fd-893f-69d1fdf73a71,microsoft/onnxscript,10075881,gramalingam,MDQ6VXNlcjEwMDc1ODgx,user,Applying suggested changes from code review,completed,2026-07-16 21:44:19.215,fa1cd93c-c349-4388-a738-379e8baaa7cb,...,Applying suggested changes from code review,sweagent-capi:gpt-5.4,"@copilot Fix the code for all comments in [this review thread](https://github.com/microsoft/onnxscript/pull/2829#pullrequestreview-4708941209).\n\nWhen a review comment includes a suggested change, apply the suggestion exactly unless the instructions in the 'Additional instructions' section below indicate otherwise.\n\nDo not make changes beyond what is described in the linked review thread, unless requested in the 'Additional instructions' section below.",completed,ai_credits,8.351408e+10,8.351408e+10,0.0,NaN,2026-07-16 21:44:21.954
2,00438e4b-08b5-4db4-92a4-cddc8be88981,microsoft/onnxruntime-genai,30328909,tianleiwu,MDQ6VXNlcjMwMzI4OTA5,user,Improving execution provider selection in CUDA CI tests,completed,2026-08-27 16:08:24.671,d849453c-48da-4787-9a3e-3c72cf353acc,...,Improving execution provider selection in CUDA CI tests,sweagent-capi:gpt-5.6-sol,"Please help improve windows cuda x64 CI:\n\nThe failing step is **Run the Python Tests** in the [Windows CUDA workflow](https://github.com/microsoft/onnxruntime-genai/blob/c5bda4a8b16db7cf3ecc5d77dd51cdcd63a536d3/.github/workflows/win-cuda-x64-build.yml#L200-L202). The Java tests passed, and the subsequent artifact listing ran only because it uses `if: always()`.\n\n### Recommended fix\n\nThis CUDA job currently lets the Python driver auto-detect execution providers. That can select unintended providers or fall back to CPU depending on when the CUDA plugin becomes visible. Make the job deterministic by explicitly requesting CUDA:\n\n```yaml\n- name: Run the Python Tests\n run: |\n python test/python/test_onnxruntime_genai.py `\n --cwd ""test\python"" `\n --test_models ""test\models"" `\n --eps cuda `\n --e2e\n```\n\nThe driver already supports `--eps cuda` in [test_onnxruntime_genai.py](https://github.com/microsoft/onnxruntime-genai/blob/c5bda4a8b16db7cf3ecc5d77dd51cdcd63a536d3/test/python/test_onnxruntime_genai.py#L89-L95), and uses it when downloading regular and graph-capture models at [lines 124–132](https://github.com/microsoft/onnxruntime-genai/blob/c5bda4a8b16db7cf3ecc5d77dd51cdcd63a536d3/test/python/test_onnxruntime_genai.py#L124-L132).\n\nAlso validate CUDA availability before downloading several models so a plugin-loading problem fails immediately:\n\n```yaml\n- name: Verify CUDA EP\n shell: pwsh\n run: |\n python -c ""import onnxruntime_genai as og; assert og.is_cuda_available(), 'CUDA EP failed to load'""\n```\n\nPlace this immediately before **Run the Python Tests**.\n\n### Separate cache correction\n\nThe sccache warning is not the test failure, but the cache is currently ineffective. `SCCACHE_DIR` is set only inside the build step, so `sccache` actually uses:\n\n```text\nC:\Users\cloudtest\AppData\Local\Mozilla\sccache\cache\n```\n\nwhile the cache action tries to save `${{ github.workspace }}\sccache`, which does not exist. Define it at workflow scope:\n\n```yaml\nenv:\n SCCACHE_DIR: ${{ github.workspace }}\sc

In [36]:
# need an aggregated table by repository_full_name
# columns required 
# repository_full_name
# count of tasks call it task_count
# count of unique creator_id, call it user_count
# count of sessions, call it session_count
# count of non-null prompts, call it prompt_count
# ratio of sessions to tasks, call it session_task_ratio
# ratio of non-null prompts to sessions, call it prompt_session_ratio
# ratio of non-null prompts to tasks, call it prompt_task_ratio
# ratio of tasks to unique users, call it task_user_ratio
# sort the data by task_count (desc), then by session_task_ratio (desc), then by prompt_session_ratio (desc), then by prompt_task_ratio (desc)
# display the top 10 repositories after sorting
# write the entire sorted table to a file as a CSV to the `output_dir` variable as `sorted_repositories.csv`
sorted_repositories = (
    task_sessions.groupby('repository_full_name', dropna=False)
    .agg(
        task_count=('task_id', 'nunique'),
        user_count=('creator_id', 'nunique'),
        session_count=('session_id', 'size'),
        prompt_count=('prompt', 'count'),
    )
    .reset_index()
)

sorted_repositories = (
    sorted_repositories.assign(
        session_task_ratio=lambda x: x.session_count / x.task_count,
        prompt_session_ratio=lambda x: x.prompt_count / x.session_count,
        prompt_task_ratio=lambda x: x.prompt_count / x.task_count,
        task_user_ratio=lambda x: x.task_count / x.user_count,
    )
    .sort_values(
        ['task_count', 'session_task_ratio', 'prompt_session_ratio', 'prompt_task_ratio'],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(sorted_repositories.head(10))
sorted_repositories.to_csv(output_dir / 'sorted_repositories.csv', index=False)


,repository_full_name,task_count,user_count,session_count,prompt_count,session_task_ratio,prompt_session_ratio,prompt_task_ratio,task_user_ratio
0,github/gh-aw,14301,14,24489,11690,1.712398,0.477357,0.817425,1021.500000
1,microsoft/vscode,2943,89,4005,698,1.360856,0.174282,0.237173,33.067416
2,dotnet/runtime,2479,99,6861,2126,2.767648,0.309867,0.857604,25.040404
3,github/gh-aw-firewall,1864,9,2420,821,1.298283,0.339256,0.440451,207.111111
4,github/gh-aw-mcpg,1863,3,2052,478,1.101449,0.232943,0.256575,621.000000
5,musicmonk42/the_code_factory_working_v2,1769,1,2044,760,1.155455,0.371820,0.429621,1769.000000
6,mlflow/mlflow,1488,13,2402,619,1.614247,0.257702,0.415995,114.461538
7,epam/ketcher,1444,10,1957,599,1.355263,0.306081,0.414820,144.400000
8,camunda/camunda,1255,100,2794,864,2.226295,0.309234,0.688446,12.550000
9,kubestellar/console,1157,1,1352,46,1.168539,0.034024,0.039758,1157.000000


In [37]:
# return the descriptive statistics for the sorted_repositories DataFrame
sorted_repositories.describe()

,task_count,user_count,session_count,prompt_count,session_task_ratio,prompt_session_ratio,prompt_task_ratio,task_user_ratio
count,40176.000000,40176.000000,40176.000000,40176.000000,40176.000000,40176.000000,40176.000000,40176.000000
mean,8.749552,1.371466,15.709404,5.676772,1.571263,0.432192,0.668318,5.685901
std,83.041180,2.436766,146.568783,66.264155,1.306355,0.427417,1.047467,24.589659
min,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000
25%,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000
50%,2.000000,1.000000,3.000000,1.000000,1.000000,0.333333,0.500000,2.000000
75%,5.000000,1.000000,8.000000,3.000000,1.750000,1.000000,1.000000,4.000000
max,14301.000000,198.000000,24489.000000,11690.000000,49.600000,1.000000,49.600000,1769.000000
